In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

slug_df = pd.read_csv('processed_EA-10.csv')
slug_df.head()

In [ ]:
slug_ts = slug_df[['TimeStamp', 'TH_Pressure']]
slug_ts['TimeStamp'] = pd.to_datetime(slug_ts['TimeStamp'])
slug_ts.set_index('TimeStamp', inplace=True)
slug_ts

In [ ]:
df = slug_ts
fig = px.scatter(df, x=slug_ts.index, y="TH_Pressure", 
                title="Whole timeserie for processed well EA-36 (Bar)")
fig.show()

In [ ]:
slug_ts_focus = slug_ts["2022-08-01 09:00:00":"2022-08-01 12:00:00"]

fig = px.scatter()
fig.add_scatter(x=slug_ts_focus.index, y=slug_ts_focus["TH_Pressure"],name="Selected timeserie for processed well EA-36 (Bar)")
fig.show()

In [ ]:
#upsampling - interpolation

from scipy.interpolate import CubicSpline

xmin = 1
xmax = len(slug_ts_focus['TH_Pressure'])-1
some_step = 0.1                         # this is a 6 seconds step, 10 points per minute
cs = CubicSpline(range(len(slug_ts_focus)), slug_ts_focus['TH_Pressure'])
x_range = np.arange(xmin, xmax, some_step)

slug_ts_focus.insert(1, "idx", np.linspace(0, len(slug_ts_focus)-1, num=len(slug_ts_focus)), True)

figure_FHP = px.scatter()
figure_FHP.add_scatter(x=x_range, y=cs(x_range) ,  name="FHP")
figure_FHP.add_scatter(x=slug_ts_focus['idx'], y=slug_ts_focus["TH_Pressure"],name="interpolated")

figure_FHP.show()

In [ ]:
fig_ss_dataanalysis, data = plt.subplots(1, 2, figsize=(17, 3))

freq_units = 1/len(slug_ts_focus)    # I need to double check the untits of the FFT
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_dataanalysis.suptitle('EA-36 THP stats')
data[0].set_xlabel('time (1m)')
data[0].set_ylabel('THP (Bar)')
data[0].tick_params(labelrotation=45)
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[1].set_yscale('log')
data[1].set_xscale('log')

fft_ss = np.fft.rfft(slug_ts_focus['TH_Pressure'])
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)

data[0].plot(slug_ts_focus['TH_Pressure'])
data[1].plot(power_spectrum_ss)
plt.plot

In [ ]:
fig_ss_analysis_interpolated, data = plt.subplots(1, 2, figsize=(17, 3))

freq_units = 1/len(cs(x_range))    # I need to double check the untits of the FFT
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_analysis_interpolated.suptitle('EA-36 THP stats August 1st - interpolated')
data[0].set_xlabel('time (6s)')
data[0].set_ylabel('THP (Bar)')
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[1].set_yscale('log')
data[1].set_xscale('log')

fft_ss = np.fft.rfft(cs(x_range))
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)

data[0].plot(cs(x_range))
data[1].plot(power_spectrum_ss)
plt.plot


From now, I am going to work on the interpolated signal. It is more likely to work well in this approach. 
Also, I am dropping timestamps because I am not good with this. 

In [ ]:
slug_Aug1st = pd.DataFrame(cs(x_range), columns = ['TH Pressure (Bar)'])

In [ ]:
#is this stationary?
from statsmodels.tsa.stattools import adfuller

def adf_test(timeseries):
    print ('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
       dfoutput['Critical Value (%s)'%key] = value
    print (dfoutput)


In [ ]:
adf_test(slug_Aug1st)

p is < 0.05 therefore the time series is stationary

In [ ]:
from statsmodels.tsa.seasonal import STL

res = STL(slug_Aug1st, period=270).fit()
res.plot()
plt.show()

In [ ]:
from statsmodels.graphics.gofplots import qqplot
qqplot(res.resid, line='s')

In [ ]:
res.resid.hist(bins=50)

In [ ]:
slug_Aug1st

In [ ]:
import statsmodels.api as sm 

fig = plt.figure(figsize=(12,8))
ax1 = fig.add_subplot(211)
fig = sm.graphics.tsa.plot_acf(slug_Aug1st["TH Pressure (Bar)"], lags=50, ax=ax1)
ax2 = fig.add_subplot(212)
fig = sm.graphics.tsa.plot_pacf(slug_Aug1st["TH Pressure (Bar)"], lags=50, ax=ax2)
plt.show()

In [ ]:
# tryin manually. Finite differencing order

from statsmodels.tsa.stattools import adfuller

results = adfuller(slug_Aug1st['TH Pressure (Bar)'].values) 
print('p-value:', results[1])

This is a pretty bad fit. There is not really a dominant frequency and this is messing up the analysis. 

In [ ]:
start_date = '2022-06-01 03:20:00'
end_date = '2022-06-01 07:39:00'

slug_ts_focus = slug_ts[start_date:end_date]

fig = px.scatter()
fig.add_scatter(x=slug_ts_focus.index, y=slug_ts_focus["TH_Pressure"],name="Selected timeserie for processed well EA-36 (Bar)")
fig.show()

In [ ]:
xmin = 1
xmax = len(slug_ts_focus['TH_Pressure'])-1
some_step = 0.1                         # this is a 6 seconds step, 10 points per minute
cs = CubicSpline(range(len(slug_ts_focus)), slug_ts_focus['TH_Pressure'])
x_range = np.arange(xmin, xmax, some_step)

slug_ts_focus.insert(1, "idx", np.linspace(0, len(slug_ts_focus)-1, num=len(slug_ts_focus)), True)

figure_FHP = px.scatter()
figure_FHP.add_scatter(x=x_range, y=cs(x_range) ,  name="FHP")
figure_FHP.add_scatter(x=slug_ts_focus['idx'], y=slug_ts_focus["TH_Pressure"],name="interpolated")

In [ ]:
ig_ss_analysis_interpolated, data = plt.subplots(1, 2, figsize=(17, 3))

freq_units = 1/len(cs(x_range))    # I need to double check the untits of the FFT
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_analysis_interpolated.suptitle('EA-36 THP stats August 1st - interpolated')
data[0].set_xlabel('time (6s)')
data[0].set_ylabel('THP (Bar)')
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[1].set_yscale('log')
data[1].set_xscale('log')

fft_ss = np.fft.rfft(cs(x_range))
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)

data[0].plot(cs(x_range))
data[1].plot(power_spectrum_ss)
plt.plot

In [ ]:
slug_Jun1st = pd.DataFrame(cs(x_range), columns = ['TH Pressure (Bar)'])

res = STL(slug_Jun1st, period=250).fit()
res.plot()
plt.show()

In [ ]:
qqplot(res.resid, line='s')

In [ ]:
res.resid.hist(bins=50)

In [ ]:
fig = plt.figure(figsize=(12,8))
ax1 = fig.add_subplot(211)
fig = sm.graphics.tsa.plot_acf(res.seasonal, lags=40, ax=ax1)
ax2 = fig.add_subplot(212)
fig = sm.graphics.tsa.plot_pacf(res.seasonal, lags=40, ax=ax2)
plt.show()

In [ ]:
import itertools

# Define the p, d and q parameters to take any value between 0 and 2
p = q = range(10, 15)
d = range(0, 1) 

# Generate all different combinations of p, q and q triplets
pdq = list(itertools.product(p, d, q))

# Generate all different combinations of seasonal p, q and q triplets
#seasonal_pdq = [(x[0], x[1], x[2], 12) for x in list(itertools.product(p, d, q))]

print('Examples of parameter combinations for Seasonal ARIMA...')
for i in pdq:
    print('SARIMAX: {} '.format(i))


In [ ]:
import warnings
#warnings.filterwarnings("ignore") # specify to ignore warning messages

for param in pdq:
   # for param_seasonal in seasonal_pdq:
    try:
    
        mod = sm.tsa.statespace.ARIMA(slug_Jun1st,
                                        order=param#,
                                        #seasonal_order=param_seasonal,
                                        #enforce_stationarity=False,
                                        #enforce_invertibility=False
                                        )
        results = mod.fit()

        print('ARIMA{} - AIC:{}'.format(param, results.aic))
    except:
        continue

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

data = slug_Jun1st
order = (2, 1, 2)
model = ARIMA(data, order, freq='D')
model = model.fit()
print (model.predict(1, 20))

